# Feeana: Dual-Head Fine-Tuning + ONNX Export (Google Colab T4)

Minimal, reproducible notebook targeting Google Colab T4 GPU runtime.

### Overview:
- **Base Model**: configurable — set once in the *Config* cell below (`bert-base-multilingual-cased` for mBERT, or the DistilXLM-R default). Public models, no HF token required.
- **Architecture**: Shared encoder + 15-way `issue` head + 3-way `polarity` head
- **Training**: Full fine-tuning of the shared encoder
- **Export**: ONNX FP32 to INT8 quantization + smoke test
- **Dependencies**: Minimal installation targeting training dependencies without touching preinstalled Colab numpy/pandas
- **Post-training evaluation**: offloaded to local CPU execution (not in this notebook)

### Step 1: Config — Select Base Model

**This is the ONE line you change** to switch which model is fine-tuned and exported.

- `bert-base-multilingual-cased` -> mBERT (output tag: `mbert`)
- `nreimers/mMiniLMv2-L12-H384-distilled-from-XLMR-Large` -> DistilXLM-R (default, tag: `distilxlmr`)

All downstream steps (training, ONNX export, packaging) read this env var automatically.

In [ ]:
import os
os.environ["FEEANA_MODEL_NAME"] = ""
print(f"[CONFIG] Base model: {os.environ['FEEANA_MODEL_NAME']}")

### Step 2: Install Dependencies

Upgrades `torchao` to `>=0.16.0` while avoiding unnecessary `-U` upgrades or version locks on preinstalled Colab packages (`numpy`, `pandas`, `torch`).

In [ ]:
!pip install -q \
  "torchao>=0.16.0" \
  "transformers>=4.38.0" \
  "datasets>=2.18.0" \
  "evaluate>=0.4.0" \
  "accelerate>=0.27.0" \
  "scikit-learn>=1.3.0" \
  "onnx>=1.15.0" \
  "onnxruntime>=1.17.0"

### Step 3: Upload & Verify Files

Upload the complete local `scripts/training/` directory into Colab so it appears at `/content/scripts/training/`. Include the Python scripts used below and the `data/` directory.

In [ ]:
from pathlib import Path
import pandas as pd

required_files = [
    Path("scripts/training/finetune.py"),
    Path("scripts/training/checkpoint_paths.py"),
    Path("scripts/training/export_model_onnx.py"),
    Path("scripts/training/data/train.csv"),
    Path("scripts/training/data/val.csv"),
    Path("scripts/training/data/test.csv"),
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

for path in required_files:
    if path.suffix == ".csv":
        print(f"[FOUND] {path} ({len(pd.read_csv(path)):,} rows)")
    else:
        print(f"[FOUND] {path}")
print("[READY] Training files verified.")

### Step 4: Fine-Tuning (Full Training)

The base model comes from the *Config* cell (`FEEANA_MODEL_NAME`), so no `--model-name` flag is needed here. Batch 16 fits the T4 for the DistilXLM-R base; if you hit a CUDA OOM with mBERT (~110M params), do NOT lower `--batch-size` to 8.

In [ ]:
!python scripts/training/finetune.py --epochs 8 --patience 8 --batch-size 16 --lr 4e-5 --seed 42

### Step 5: Decision Gate — Compare vs Baseline [Archived]

This step was used during the DistilXLM-R learning-rate sweep (4e-5 vs 3e-5 vs 2e-5) and has
concluded. It is no longer active and will be skipped during Run All. The original code
is retained below for reference.

In [ ]:
print("[INFO] Step 5 is archived — skipping.")
# Original code retained for reference
# import json
# from pathlib import Path
#
# # 2e-5 baseline (8-epoch run, seed 42)
# BASELINE_LR = "2e-5"
# BASELINE_BEST_VAL_ISSUE_F1 = 0.7650376210750968
# BASELINE_BEST_VAL_POLARITY_F1 = 0.8539
#
# # Load the most recent 4e-5 training report
# reports = sorted(
#     Path("scripts/training/reports").glob("distilxlmr_training_run_*.json")
# )
# if not reports:
#     raise FileNotFoundError("No training report found. Check that finetune.py ran successfully.")
#
# report_path = reports[-1]
# with open(report_path, encoding="utf-8") as f:
#     report = json.load(f)
#
# lr_used = report.get("hyperparameters", {}).get("lr", "unknown")
# best_val_issue_f1 = report.get("best_val_issue_macro_f1", 0.0)
# epochs_completed = report.get("hyperparameters", {}).get("epochs_completed", 0)
# epoch_logs = report.get("epoch_logs", [])
#
# best_ep = max(epoch_logs, key=lambda x: x.get("val_issue_macro_f1", 0.0), default={})
# best_val_polarity_f1 = best_ep.get("val_polarity_macro_f1", 0.0)
#
# print(f"Report: {report_path.name}")
# print(f"LR: {lr_used} | Epochs completed: {epochs_completed}")
# print()
#
# # Per-epoch validation table
# header = f"{'Epoch':>5}  {'val_issue_F1':>12}  {'val_polarity_F1':>15}"
# print(header)
# print("-" * len(header))
# for ep in epoch_logs:
#     print(
#         f"{ep['epoch']:>5}  "
#         f"{ep['val_issue_macro_f1']:>12.4f}  "
#         f"{ep['val_polarity_macro_f1']:>15.4f}"
#     )
# print()
#
# # Head-to-head comparison
# print("Metric                          4e-5 (this run)   2e-5 baseline")
# print("-" * 66)
# print(
#     f"Best val issue macro-F1       {best_val_issue_f1:>16.4f}"
#     f"   {BASELINE_BEST_VAL_ISSUE_F1:>16.4f}"
# )
# print(
#     f"Best val polarity macro-F1    {best_val_polarity_f1:>16.4f}"
#     f"   {BASELINE_BEST_VAL_POLARITY_F1:>16.4f}"
# )
# print()
#
# # Verdict
# if best_val_issue_f1 > BASELINE_BEST_VAL_ISSUE_F1:
#     delta = best_val_issue_f1 - BASELINE_BEST_VAL_ISSUE_F1
#     print("=" * 66)
#     print("  PROCEED — LR 4e-5 improved val issue macro-F1 by")
#     print(f"  +{delta:.4f} over the {BASELINE_LR} baseline. Continue to export.")
#     print("=" * 66)
# else:
#     delta = BASELINE_BEST_VAL_ISSUE_F1 - best_val_issue_f1
#     print("=" * 66)
#     print("  ABORT — LR 4e-5 did NOT improve val issue macro-F1.")
#     print(f"  It trailed the {BASELINE_LR} baseline by {delta:.4f}.")
#     print(f"  Settle with the current {BASELINE_LR} checkpoint.")
#     print("  You may still continue export below to inspect pipeline output.")
#     print("=" * 66)

### Step 6: Export ONNX & Quantize

Exports the trained checkpoint to FP32 ONNX, retains the FP32 intermediate, and creates the required INT8 post-training quantized artifact.

In [ ]:
!python scripts/training/export_model_onnx.py \
    --out-dir scripts/training/exports \
    --keep-fp32

### Step 7: Package Outputs (ZIP)

Collects the trained checkpoint, ONNX exports, and the training report into a single
zip archive. Archive name and folder paths resolve dynamically from the active model tag
(set in Step 1). No auto-download is triggered — download manually from the files panel.

In [ ]:
from pathlib import Path
import shutil
import os
import sys

# Import shared tag resolver
sys.path.insert(0, "scripts/training")
from checkpoint_paths import resolve_tag, DEFAULT_MODEL_NAME

model_name = os.environ.get("FEEANA_MODEL_NAME") or DEFAULT_MODEL_NAME
tag = resolve_tag(model_name)

ZIP_ROOT = f"feeana-{tag}-output"
staging_root = Path(ZIP_ROOT)
if staging_root.exists():
    shutil.rmtree(staging_root)
staging_root.mkdir(parents=True)

# 1. Checkpoint + label mappings
ckpt_dir = Path(f"scripts/training/checkpoints/{tag}")
for name in ("best_model.pt", "label_mappings.json"):
    src = ckpt_dir / name
    if src.exists():
        shutil.copy2(src, staging_root / name)

# 2. ONNX exports + tokenizer/config
export_dir = Path(f"scripts/training/exports/{tag}")
for name in (
    "fp32.onnx",
    "int8.onnx",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "config.json",
    "label_mappings.json",
):
    src = export_dir / name
    if src.exists():
        shutil.copy2(src, staging_root / name)

# 3. Training report
reports = Path("scripts/training/reports")
report_files = {}
training_reports = sorted(reports.glob(f"{tag}_training_run_*.json"))
if training_reports:
    report_files["training_run.json"] = training_reports[-1]

for dest_name, src in report_files.items():
    if src.exists():
        shutil.copy2(src, staging_root / dest_name)

if not list(staging_root.iterdir()):
    raise FileNotFoundError(f"No artifacts found for tag '{tag}' to package. Ensure training and export ran successfully.")

archive_path = Path(
    shutil.make_archive(
        ZIP_ROOT,
        "zip",
        root_dir=".",
        base_dir=staging_root.name,
    )
)
shutil.rmtree(staging_root)

print(f"[PACKAGED] {archive_path}")
print(f"[READY] Zip created with artifacts inside {ZIP_ROOT}/. Download it manually from the files panel.")